<a href="https://colab.research.google.com/github/Cautioncrazy/BleachSoulDeck/blob/main/kohya_ss_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Train with Kohya's Stable Diffusion Trainers
%cd /content

from google.colab import drive
drive.mount('/content/drive')

!pip install dadaptation==3.1 diffusers[torch]==0.17.1 easygui==0.98.3 einops==0.6.0 fairscale==0.4.13 ftfy==6.1.1 gradio==3.36.1 huggingface-hub==0.14.1
!pip install lion-pytorch==0.0.6 lycoris_lora==1.8.0.dev6 open-clip-torch==2.20.0 prodigyopt==1.0 pytorch-lightning==1.9.0 safetensors==0.3.1 timm==0.6.12
!pip install tk==0.1.0 transformers==4.30.2 voluptuous==0.13.1 wandb==0.15.0 xformers==0.0.20 omegaconf

%cd /content
!git clone -b 0.41.0 https://github.com/TimDettmers/bitsandbytes
%cd /content/bitsandbytes
!CUDA_VERSION=118 make cuda11x
!python setup.py install

%cd /content
!git clone -b v1.0 https://github.com/camenduru/kohya_ss
%cd /content/kohya_ss

!python kohya_gui.py --share --headless

In [ ]:
#@title Convert Safetensors to Diffusers
from_safetensors_url = '' #@param {type:"string"}
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/v0.17.1/scripts/convert_original_stable_diffusion_to_diffusers.py
!wget {from_safetensors_url} -O /content/model.safetensors
!python3 convert_original_stable_diffusion_to_diffusers.py --half --from_safetensors --checkpoint_path model.safetensors --dump_path /content/model

In [ ]:
#@title Push to HF.co

import os
from huggingface_hub import create_repo, upload_folder

hf_token = 'HF_WRITE_TOKEN' #@param {type:"string"}
repo_id = 'username/reponame' #@param {type:"string"}
commit_message = '\u2764' #@param {type:"string"}
create_repo(repo_id, private=True, token=hf_token)
model_path = '/content/train/model' #@param {type:"string"}
upload_folder(folder_path=f'{model_path}', path_in_repo='', repo_id=repo_id, commit_message=commit_message, token=hf_token)

In [ ]:
#@title Push to DagsHub.com

!pip -q install dagshub
from dagshub.upload import Repo, create_repo

repo_id = 'reponame' #@param {type:"string"}
org_name = 'orgname' #@param {type:"string"}
commit_message = '\u2764' #@param {type:"string"}
create_repo(f"{repo_id}", org_name=f"{org_name}")
repo = Repo(f"{org_name}", f"{repo_id}")
model_path = '/content/train/model' #@param {type:"string"}
repo.upload("/content/models", remote_path="data", commit_message=f"{commit_message}", force=True)

# Task
Set up and launch the Kohya_ss GUI for LoRA training by preparing training directories, organizing images with captions, configuring hyperparameters, and executing the web interface script.

## Prepare Training Directories

### Subtask:
Create the required folder structure for LoRA training, including directories for images, logging, and model output.


**Reasoning**:
Create the required directory structure for LoRA training to organize images, logs, and model outputs using the os module.



In [1]:
import os

# Define the root and subdirectories
root_dir = '/content/lora_training'
sub_dirs = ['img', 'log', 'model']

# Create directories
for sub_dir in sub_dirs:
    path = os.path.join(root_dir, sub_dir)
    os.makedirs(path, exist_ok=True)

print(f'Training directory structure created at: {root_dir}')
print('Subdirectories:', os.listdir(root_dir))

Training directory structure created at: /content/lora_training
Subdirectories: ['img', 'model', 'log']


## Image Preparation Script

### Subtask:
Provide a script to help organize and rename training images with appropriate captions in the expected Kohya format.


**Reasoning**:
I will provide a Python script that defines a specific Kohya-compliant directory (e.g., '100_mysubject'), creates it within the 'img' directory, and adds placeholder logic for organizing images and their corresponding caption files.



In [2]:
import os

# 1. Define folder name using Kohya convention: [repeats]_[identifier]
images_dir = '/content/lora_training/img'
subject_folder = '100_mysubject'
target_path = os.path.join(images_dir, subject_folder)

# 2. Create the subdirectory
os.makedirs(target_path, exist_ok=True)

# 3. Placeholder for image organization
# In a real scenario, you would upload files here.
# For this script, we'll create dummy files to demonstrate the structure.

example_images = ['image1.jpg', 'image2.png']
for img_name in example_images:
    # Create a dummy image file
    img_file = os.path.join(target_path, img_name)
    with open(img_file, 'w') as f:
        f.write('dummy image data')

    # 4. Ensure a corresponding caption text file exists
    caption_file = os.path.splitext(img_file)[0] + '.txt'
    with open(caption_file, 'w') as f:
        f.write('mysubject, high quality, 4k')

# 5. Print final directory structure to verify
print(f'Target directory: {target_path}')
print('Files in directory:', os.listdir(target_path))

Target directory: /content/lora_training/img/100_mysubject
Files in directory: ['image1.jpg', 'image2.txt', 'image1.txt', 'image2.png']


## Configure Training Parameters

### Subtask:
Generate a configuration cell to set base model paths and training hyperparameters specific to LoRA training.


**Reasoning**:
I will create a configuration cell that defines the base model paths, training hyperparameters, and directory paths for LoRA training as specified in the instructions.



In [3]:
#@title LoRA Training Configuration

# 1. Define base model path
# Using standard SD v1.5 path as an example, update if using a custom converted model
pretrained_model_name_or_path = "/content/model"

# 2. Set Training Hyperparameters
learning_rate = 1e-4
resolution = 512
batch_size = 1
max_train_steps = 2000
network_module = "networks.lora"
network_dim = 32
network_alpha = 16

# 3. Specify Directory Paths (using variables from previous steps)
# root_dir = '/content/lora_training' was defined earlier
train_data_dir = "/content/lora_training/img"
output_dir = "/content/lora_training/model"
logging_dir = "/content/lora_training/log"

# 4. Summary of Configuration
print("--- LoRA Training Configuration Summary ---")
print(f"Base Model: {pretrained_model_name_or_path}")
print(f"Train Data Dir: {train_data_dir}")
print(f"Output Dir: {output_dir}")
print(f"Logging Dir: {logging_dir}")
print(f"Learning Rate: {learning_rate}")
print(f"Resolution: {resolution}")
print(f"Batch Size: {batch_size}")
print(f"Max Steps: {max_train_steps}")
print("-------------------------------------------")

--- LoRA Training Configuration Summary ---
Base Model: /content/model
Train Data Dir: /content/lora_training/img
Output Dir: /content/lora_training/model
Logging Dir: /content/lora_training/log
Learning Rate: 0.0001
Resolution: 512
Batch Size: 1
Max Steps: 2000
-------------------------------------------


## Launch Web Interface

### Subtask:
Execute the Kohya_ss GUI launch command with the necessary arguments to access the training dashboard via a public URL.


**Reasoning**:
I will execute the `kohya_gui.py` script using Python, ensuring the working directory is set to the cloned repository. I will include the `--share` and `--headless` flags as instructed to generate a public URL for remote access within the Colab environment.

**Reasoning**:
I will provide the code block to change the directory to 'kohya_ss' and execute the 'kohya_gui.py' script with the required '--share' and '--headless' flags to launch the web interface.



In [4]:
import os

# Navigate to the kohya_ss directory and launch the GUI
%cd /content/kohya_ss

# Execute the script with necessary flags for Colab
!python kohya_gui.py --share --headless

[Errno 2] No such file or directory: '/content/kohya_ss'
/content
python3: can't open file '/content/kohya_gui.py': [Errno 2] No such file or directory


**Reasoning**:
The previous attempt to launch the GUI failed because the directory '/content/kohya_ss' was not found. I will re-run the installation and cloning steps from the first cell to ensure the repository is correctly set up before attempting to launch the GUI again.



In [5]:
import os

# Ensure we are in /content
%cd /content

# Clone the repository if it doesn't exist
if not os.path.exists('/content/kohya_ss'):
    !git clone -b v1.0 https://github.com/camenduru/kohya_ss

# Navigate to the directory and ensure we are there
%cd /content/kohya_ss

# Launch the GUI with the required flags
!python kohya_gui.py --share --headless

/content
Cloning into 'kohya_ss'...
remote: Enumerating objects: 8911, done.
remote: Total 8911 (delta 0), reused 0 (delta 0), pack-reused 8911 (from 1)
Receiving objects: 100% (8911/8911), 6.18 MiB | 11.43 MiB/s, done.
Resolving deltas: 100% (6190/6190), done.
/content/kohya_ss
/content/kohya_ss/library/common_gui.py:793: SyntaxWarning: invalid escape sequence '\i'
  log.error(f"The provided path '{folder_path}' is not a valid folder. Please follow the folder structure documentation found at docs\image_folder_structure.md ...")
/content/kohya_ss/library/common_gui.py:816: SyntaxWarning: invalid escape sequence '\i'
  log.error(f"Please follow the folder structure documentation found at docs\image_folder_structure.md ...")
/content/kohya_ss/library/common_gui.py:821: SyntaxWarning: invalid escape sequence '\i'
  log.error(f"No image folders found in {folder_path}. Please follow the folder structure documentation found at docs\image_folder_structure.md ...")
Traceback (most recent call 

**Reasoning**:
The previous execution failed due to a 'ModuleNotFoundError: No module named easygui', despite the initial installation cell. I will re-install the missing dependencies, specifically 'easygui', and then launch the Kohya_ss GUI using the required flags.



In [6]:
!pip install easygui

import os
%cd /content/kohya_ss

# Launch the GUI with the required flags for Colab
!python kohya_gui.py --share --headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.7/92.7 kB 5.3 MB/s eta 0:00:00
/content/kohya_ss
22:34:15-944930 INFO     headless: True                                         
22:34:15-947959 INFO     Load CSS...                                            
/content/kohya_ss/kohya_gui.py:37: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  interface = gr.Blocks(
/content/kohya_ss/kohya_gui.py:37: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  interface = gr.Blocks(
╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /content/kohya_ss/kohya_gui.py:133 in <module>                               │
│                                                                              │
│   132                                                              

**Reasoning**:
The Gradio error 'AttributeError: module gradio has no attribute Box' indicates a version mismatch, as 'gr.Box' was replaced by 'gr.Container' or 'gr.Group' in newer Gradio versions. Since the current environment has Gradio 3.36.1 (which should have Box), the error suggests a conflict or a need for a specific version. I will downgrade Gradio to 3.36.1 explicitly to fix the UI compatibility issue.



In [7]:
!pip install gradio==3.36.1

import os
%cd /content/kohya_ss

# Launch the GUI with the required flags
!python kohya_gui.py --share --headless

INFO: pip is looking at multiple versions of mdit-py-plugins to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of mdit-py-plugins to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.5/84.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: markdown-it-py
    Found existing installation: markdown-it-py 4.0.0
    Uninstalling markdown-it-py-4.0.0:
      Successfully uninstalled markdown-it-py-4.0.0
  Attempting uninstall: mdit

# Task
Set up and launch the Kohya_ss GUI for LoRA training by preparing training directories, organizing images with captions, configuring hyperparameters, and executing the web interface script.

## Prepare Training Directories

### Subtask:
Create the required folder structure for LoRA training, including directories for images, logging, and model output.


**Reasoning**:
I will create a code block that implements the directory creation logic using the `os` module as per the instructions.



In [8]:
import os

# 1. Define base directory path
base_dir = "/content/lora_training"

# 2. Define subdirectories
sub_dirs = ["img", "log", "model"]

# 3. Create directories using a loop
for sub_dir in sub_dirs:
    path = os.path.join(base_dir, sub_dir)
    os.makedirs(path, exist_ok=True)

# 4. Verify the structure
if os.path.exists(base_dir):
    print(f"Successfully created base directory: {base_dir}")
    print("Contents:", os.listdir(base_dir))
else:
    print("Failed to create directory structure.")

Successfully created base directory: /content/lora_training
Contents: ['img', 'model', 'log']


## Image Preparation Script

### Subtask:
Provide a script to help organize and rename training images with appropriate captions in the expected Kohya format.


**Reasoning**:
I will create a Python script that implements the image organization and captioning logic, following the specific folder naming conventions and directory structure required for Kohya_ss LoRA training.



In [9]:
import os

# 1. Define folder name using Kohya convention: [repeats]_[identifier]
images_dir = "/content/lora_training/img"
subject_folder = "100_mysubject"
target_path = os.path.join(images_dir, subject_folder)

# 2. Create the subdirectory if it doesn't exist
os.makedirs(target_path, exist_ok=True)

# 3. Logic to simulate image organization and captioning
# In a real scenario, you would move uploaded files here.
# For this script, we create sample files to demonstrate the logic.

sample_images = ["image1.jpg", "image2.png"]
trigger_words = "mysubject, high quality, 4k"

for img_name in sample_images:
    img_file_path = os.path.join(target_path, img_name)

    # Simulate placing an image file (if it doesn't exist)
    if not os.path.exists(img_file_path):
        with open(img_file_path, "w") as f:
            f.write("dummy image data")

    # 4 & 5. Create a corresponding .txt caption file for each image
    caption_file_path = os.path.splitext(img_file_path)[0] + ".txt"
    with open(caption_file_path, "w") as f:
        f.write(trigger_words)

# 6. Print final directory structure to verify pairing
print(f"Target directory created: {target_path}")
print("Contents of the directory:")
for file in sorted(os.listdir(target_path)):
    print(f" - {file}")

Target directory created: /content/lora_training/img/100_mysubject
Contents of the directory:
 - image1.jpg
 - image1.txt
 - image2.png
 - image2.txt


## Configure Training Parameters

### Subtask:
Generate a configuration cell to set base model paths and training hyperparameters specific to LoRA training.


**Reasoning**:
I will create a configuration cell that defines the base model paths, training hyperparameters, and directory paths for LoRA training as specified in the instructions.



In [10]:
#@title LoRA Training Configuration

# 1. Define base model path
pretrained_model_name_or_path = "/content/model"

# 2. Set Training Hyperparameters
learning_rate = 1e-4
resolution = 512
batch_size = 1
max_train_steps = 2000
network_module = "networks.lora"
network_dim = 32
network_alpha = 16

# 3. Specify Directory Paths
train_data_dir = "/content/lora_training/img"
output_dir = "/content/lora_training/model"
logging_dir = "/content/lora_training/log"

# 4. Summary of Configuration
print("--- LoRA Training Configuration Summary ---")
print(f"Base Model: {pretrained_model_name_or_path}")
print(f"Train Data Dir: {train_data_dir}")
print(f"Output Dir: {output_dir}")
print(f"Logging Dir: {logging_dir}")
print(f"Learning Rate: {learning_rate}")
print(f"Resolution: {resolution}")
print(f"Batch Size: {batch_size}")
print(f"Max Steps: {max_train_steps}")
print(f"Network Dim: {network_dim}")
print(f"Network Alpha: {network_alpha}")
print("-------------------------------------------")

--- LoRA Training Configuration Summary ---
Base Model: /content/model
Train Data Dir: /content/lora_training/img
Output Dir: /content/lora_training/model
Logging Dir: /content/lora_training/log
Learning Rate: 0.0001
Resolution: 512
Batch Size: 1
Max Steps: 2000
Network Dim: 32
Network Alpha: 16
-------------------------------------------


## Launch Web Interface on Custom Port

### Subtask:
Execute the Kohya_ss GUI launch command with specific flags to ensure a successful connection and avoid port conflicts in the Colab environment.


**Reasoning**:
I will create a code block that navigates to the kohya_ss directory and executes the GUI script with the specific port, share, and headless flags as instructed.



In [11]:
import os

# 1. Change the current working directory
os.chdir('/content/kohya_ss')

# 2 & 3. Execute the script with specific flags: port 7861, share, and headless
!python kohya_gui.py --server_port 7861 --share --headless

/usr/local/lib/python3.12/dist-packages/gradio_client/documentation.py:106: UserWarning: Could not get documentation group for <class 'gradio.mix.Parallel'>: No known documentation group for module 'gradio.mix'
  warnings.warn(f"Could not get documentation group for {cls}: {exc}")
/usr/local/lib/python3.12/dist-packages/gradio_client/documentation.py:106: UserWarning: Could not get documentation group for <class 'gradio.mix.Series'>: No known documentation group for module 'gradio.mix'
  warnings.warn(f"Could not get documentation group for {cls}: {exc}")
22:38:44-929217 INFO     headless: True                                         
22:38:44-932320 INFO     Load CSS...                                            
Running on local URL:  http://127.0.0.1:7861
Running on public URL: https://34007e702d072f78a8.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


# Task
Set up and launch the Kohya_ss GUI for LoRA training by downloading the "misri/hassakuXLIllustrious_v30" model to "/content", preparing training directories, organizing images with captions in the "100_mysubject" format, configuring hyperparameters, and executing the web interface script.

## Download Base Model

### Subtask:
Download the Hassaku XL Illustrious v3.0 model from Hugging Face and save it to the `/content` directory.


**Reasoning**:
I will use the `huggingface_hub` library to download the specific model file 'hassakuXLIllustrious_v30.safetensors' from the repository 'misri/hassakuXLIllustrious_v30'. I will set the `local_dir` to `/content` to ensure it is accessible for the training process and verify its presence by listing the directory contents.

### Reasoning
I will use the `huggingface_hub` library to download the specific model file 'hassakuXLIllustrious_v30.safetensors' from the repository 'misri/hassakuXLIllustrious_v30'. I will set the `local_dir` to `/content` to ensure it is accessible for the training process and verify its presence by listing the directory contents.

**Reasoning**:
I will provide the Python code to download the 'hassakuXLIllustrious_v30.safetensors' model from Hugging Face to the /content directory and verify the download.



In [12]:
from huggingface_hub import hf_hub_download
import os

# 1 & 2. Download the model file from the repository
repo_id = "misri/hassakuXLIllustrious_v30"
filename = "hassakuXLIllustrious_v30.safetensors"

print(f"Downloading {filename} from {repo_id}...")
downloaded_path = hf_hub_download(
    repo_id=repo_id,
    filename=filename,
    local_dir="/content"
)

# 4. Verify the download by listing files in /content
print(f"Download complete. File saved at: {downloaded_path}")
print("Files in /content:", os.listdir("/content"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


hassakuXLIllustrious_v30.safetensors:   0%|          | 0.00/6.94G [00:00<?, ?B/s]

Download complete. File saved at: /content/hassakuXLIllustrious_v30.safetensors
Files in /content: ['.config', 'kohya_ss', '.cache', 'lora_training', 'hassakuXLIllustrious_v30.safetensors', 'sample_data']


# Task
Set up and launch the Kohya_ss GUI for LoRA training by downloading the "misri/hassakuXLIllustrious_v30" model to "/content", preparing the required training directories (`/content/lora_training/img`, `log`, `model`), organizing training images with captions in the "100_mysubject" folder format, configuring optimal hyperparameters, and executing the web interface script using the `--server_port 7861`, `--share`, and `--headless` flags.

## Prepare Training Directories

### Subtask:
Create the required folder structure for LoRA training, including directories for images, logging, and model output.


**Reasoning**:
I will create a Python code block to establish the directory structure required for LoRA training, following the specified instructions to use the `os` module, loops, and verification via printing.



In [13]:
import os

# 1. Define base directory path
base_dir = "/content/lora_training"

# 2. Define required subdirectories
sub_dirs = ["img", "log", "model"]

# 3. Create directories using a loop
for sub_dir in sub_dirs:
    path = os.path.join(base_dir, sub_dir)
    os.makedirs(path, exist_ok=True)

# 4. Verify the structure
if os.path.exists(base_dir):
    print(f"Successfully created base directory: {base_dir}")
    print("Contents:", os.listdir(base_dir))
else:
    print("Failed to create directory structure.")

Successfully created base directory: /content/lora_training
Contents: ['img', 'model', 'log']


## Image Preparation Script

### Subtask:
Provide a script to organize training images and generate captions in the '100_mysubject' format.


**Reasoning**:
I will create a Python script that implements the image organization and captioning logic, following the specific folder naming conventions and directory structure required for Kohya_ss LoRA training as specified in the instructions.



In [14]:
import os

# 1. Define the specific target path for images
target_path = '/content/lora_training/img/100_mysubject'

# 2. Use the os module to create this directory if it doesn't already exist
os.makedirs(target_path, exist_ok=True)

# 3. Implement a script that iterates through a list of sample image filenames
sample_images = ['image1.jpg', 'image2.png']
trigger_words = 'mysubject, high quality, 4k'

for img_name in sample_images:
    img_file_path = os.path.join(target_path, img_name)

    # Simulate placing an image file
    with open(img_file_path, 'w') as f:
        f.write('dummy image data')

    # 4 & 5. Create a matching text file with .txt extension and write captions
    caption_file_path = os.path.splitext(img_file_path)[0] + '.txt'
    with open(caption_file_path, 'w') as f:
        f.write(trigger_words)

# 6. Print the final directory structure to verify pairings
print(f'Target directory: {target_path}')
print('Files in directory:')
for file in sorted(os.listdir(target_path)):
    print(f' - {file}')

Target directory: /content/lora_training/img/100_mysubject
Files in directory:
 - image1.jpg
 - image1.txt
 - image2.png
 - image2.txt


## Configure Training Parameters

### Subtask:
Generate a configuration cell that points to the 'hassakuXLIllustrious_v30.safetensors' model and sets optimal hyperparameters.


**Reasoning**:
I will create a configuration cell that defines the path to the Hassaku XL model, sets the optimal training hyperparameters for an XL model (resolution 1024), and maps the training directory structure, followed by a summary print statement.



In [15]:
#@title LoRA Training Configuration (XL Model)

# 1. Define base model path to the downloaded Hassaku XL model
pretrained_model_name_or_path = "/content/hassakuXLIllustrious_v30.safetensors"

# 2. Set Optimal Training Hyperparameters for XL LoRA
learning_rate = 1e-4
resolution = 1024
batch_size = 1
max_train_steps = 2000
network_module = "networks.lora"
network_dim = 32
network_alpha = 16

# 3. Specify Directory Paths mapped to existing structure
train_data_dir = "/content/lora_training/img"
output_dir = "/content/lora_training/model"
logging_dir = "/content/lora_training/log"

# 4. Summary of Configuration
print("--- LoRA Training Configuration Summary (XL) ---")
print(f"Base Model Path: {pretrained_model_name_or_path}")
print(f"Train Data Dir: {train_data_dir}")
print(f"Output Dir: {output_dir}")
print(f"Logging Dir: {logging_dir}")
print(f"Learning Rate: {learning_rate}")
print(f"Resolution: {resolution}")
print(f"Batch Size: {batch_size}")
print(f"Network Dim: {network_dim}")
print(f"Network Alpha: {network_alpha}")
print("-------------------------------------------")

--- LoRA Training Configuration Summary (XL) ---
Base Model Path: /content/hassakuXLIllustrious_v30.safetensors
Train Data Dir: /content/lora_training/img
Output Dir: /content/lora_training/model
Logging Dir: /content/lora_training/log
Learning Rate: 0.0001
Resolution: 1024
Batch Size: 1
Network Dim: 32
Network Alpha: 16
-------------------------------------------


## Restart Kohya GUI

### Subtask:
Execute the Kohya_ss GUI launch command using port 7861 with the '--share' and '--headless' flags to ensure it is running and accessible.


### Reasoning
I will create a code block that imports the `os` module, changes the working directory to `/content/kohya_ss`, and then executes the `kohya_gui.py` script using shell commands with the `--server_port 7861`, `--share`, and `--headless` flags as specified.

**Reasoning**:
I will provide the code block that performs the specified steps: importing the 'os' module, changing the directory to '/content/kohya_ss', and executing the 'kohya_gui.py' script with the required flags to launch the web interface on port 7861.



In [16]:
import os

# 1. Import the os module
# 2. Change the current working directory to /content/kohya_ss
os.chdir('/content/kohya_ss')

# 3. Run the kohya_gui.py script with specified flags
!python kohya_gui.py --server_port 7861 --share --headless

# 4. Monitor the output above for the generated Gradio public URL

/usr/local/lib/python3.12/dist-packages/gradio_client/documentation.py:106: UserWarning: Could not get documentation group for <class 'gradio.mix.Parallel'>: No known documentation group for module 'gradio.mix'
  warnings.warn(f"Could not get documentation group for {cls}: {exc}")
/usr/local/lib/python3.12/dist-packages/gradio_client/documentation.py:106: UserWarning: Could not get documentation group for <class 'gradio.mix.Series'>: No known documentation group for module 'gradio.mix'
  warnings.warn(f"Could not get documentation group for {cls}: {exc}")
22:51:12-926481 INFO     headless: True                                         
22:51:12-929649 INFO     Load CSS...                                            
Running on local URL:  http://127.0.0.1:7861
Running on public URL: https://eb4aa961b323501ddb.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


# Task
Set up and launch the Kohya_ss GUI for Hassaku XL LoRA training by utilizing the "misri/hassakuXLIllustrious_v30" model at "/content/hassakuXLIllustrious_v30.safetensors". Ensure the training directory structure is ready at "/content/lora_training/img", "/content/lora_training/log", and "/content/lora_training/model", with images organized in the "100_mysubject" folder format. Finally, restart the web interface by navigating to "/content/kohya_ss" and executing `kohya_gui.py` using the flags `--server_port 7861`, `--share`, and `--headless` to provide a new Gradio public URL.

## Restart Kohya GUI

### Subtask:
Navigate to the /content/kohya_ss directory and execute the kohya_gui.py script using port 7861 with the --share and --headless flags to re-establish the web interface.


**Reasoning**:
I will provide the Python code to navigate to the 'kohya_ss' directory and execute the 'kohya_gui.py' script with the specified port, share, and headless flags to restart the web interface.



# Task
Terminate any running Python processes related to `kohya_gui.py` or `kohya_ss`, then restart the Kohya_ss GUI by navigating to "/content/kohya_ss" and executing the launch command with the `--server_port 7861`, `--share`, and `--headless` flags to provide a new Gradio public URL for LoRA training with the "misri/hassakuXLIllustrious_v30" model.

## Terminate Existing Kohya Processes

### Subtask:
Search for and kill any running Python processes related to 'kohya_gui.py' or 'kohya_ss' to ensure the port and memory are fully cleared.


**Reasoning**:
I will provide the reasoning and code to identify and terminate any existing Kohya processes to free up system resources and the network port.



In [ ]:
import os
import signal
import subprocess

# 1. Identify Kohya-related processes
# We use pgrep to find PIDs of processes matching 'kohya_gui' or 'kohya_ss'
try:
    # Using subprocess to capture the output of the ps/grep command
    p1 = subprocess.Popen(['ps', 'aux'], stdout=subprocess.PIPE)
    p2 = subprocess.Popen(['grep', '-E', 'kohya_gui|kohya_ss'], stdin=p1.stdout, stdout=subprocess.PIPE)
    p3 = subprocess.Popen(['grep', '-v', 'grep'], stdin=p2.stdout, stdout=subprocess.PIPE)
    p1.stdout.close()
    p2.stdout.close()
    output, _ = p3.communicate()

    lines = output.decode().strip().split('\n')
    pids = [int(line.split()[1]) for line in lines if line]

    if not pids:
        print("No existing Kohya processes found.")
    else:
        print(f"Found PIDs to terminate: {pids}")
        # 2. Terminate identified processes
        for pid in pids:
            try:
                os.kill(pid, signal.SIGKILL)
                print(f"Successfully terminated process {pid}")
            except ProcessLookupError:
                print(f"Process {pid} already terminated.")
            except Exception as e:
                print(f"Could not kill process {pid}: {e}")

        print("All targeted Kohya processes have been cleared.")

except Exception as e:
    print(f"An error occurred while trying to clear processes: {e}")